# ⚡🔥 Electricity & Gas Pricing Rates Analysis

This notebook analyzes and visualizes **Usage Rates** ($/kWh for Electricity, $/MJ for Gas) and **Daily Supply Rates** ($/day) from consolidated utility bills (2024 - 2026).

It features interactive **Plotly figures** with hover tooltips showing exact rates, billing date ranges, providers, total usage, and bill costs.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Add project root directory to sys.path
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
from src.config import OUTPUT_DIR, CONSOLIDATED_CSV_PATH

# Output directory
output_dir = OUTPUT_DIR
consolidated_csv_path = CONSOLIDATED_CSV_PATH

## 📂 1. Load and Prepare Data

In [ ]:
# Load data
df = pd.read_csv(consolidated_csv_path)
df['Period Start'] = pd.to_datetime(df['Period Start'], format='mixed')
df['Period End'] = pd.to_datetime(df['Period End'], format='mixed')

# Separate Electricity and Gas
elec_df = df[df['Utility'] == 'Electricity'].sort_values('Period Start').reset_index(drop=True)
gas_df = df[df['Utility'] == 'Gas'].sort_values('Period Start').reset_index(drop=True)

# Helper function to build step timeline DataFrame for rate visualization
def build_timeline(sub_df, rate_col):
    dates = []
    rates = []
    providers = []
    p_starts = []
    p_ends = []
    for idx, row in sub_df.iterrows():
        dates.extend([row['Period Start'], row['Period End']])
        rates.extend([row[rate_col], row[rate_col]])
        providers.extend([row['Provider'], row['Provider']])
        p_starts.extend([row['Period Start'].strftime('%Y-%m-%d'), row['Period Start'].strftime('%Y-%m-%d')])
        p_ends.extend([row['Period End'].strftime('%Y-%m-%d'), row['Period End'].strftime('%Y-%m-%d')])
    return pd.DataFrame({
        'Date': dates,
        'Rate': rates,
        'Provider': providers,
        'Period Start': p_starts,
        'Period End': p_ends
    })

print(f"Loaded {len(elec_df)} electricity records and {len(gas_df)} gas records.")

## 📊 2. Comprehensive 2x2 Overview Dashboard (Interactive)

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Electricity Usage Rate ($/kWh)",
        "Gas Usage Rate ($/MJ)",
        "Electricity Daily Supply Rate ($/day)",
        "Gas Daily Supply Rate ($/day)"
    ),
    vertical_spacing=0.12, horizontal_spacing=0.08
)

elec_color = '#1f77b4' # Blue
gas_color = '#d62728'  # Red

# --- Top Left: Electricity Usage Rate ---
elec_tl = build_timeline(elec_df, 'Usage Rate ($/unit)')
fig.add_trace(
    go.Scatter(x=elec_tl['Date'], y=elec_tl['Rate'], mode='lines', line=dict(color=elec_color, width=2.5), name='Elec Usage Rate', hoverinfo='skip'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=elec_df['Period Start'], y=elec_df['Usage Rate ($/unit)'],
        mode='markers', marker=dict(color=elec_color, size=8),
        name='Elec Bill Data Point',
        customdata=np.stack((elec_df['Provider'], elec_df['Period Start'].dt.strftime('%b %d, %Y'), elec_df['Period End'].dt.strftime('%b %d, %Y'), elec_df['Total Usage'], elec_df['Total ($)']), axis=-1),
        hovertemplate="<b>Electricity Bill</b><br><b>Provider:</b> %{customdata[0]}<br><b>Period:</b> %{customdata[1]} to %{customdata[2]}<br><b>Usage Rate:</b> $%{y:.4f}/kWh<br><b>Total Usage:</b> %{customdata[3]} kWh<br><b>Total Bill:</b> $%{customdata[4]:.2f}<extra></extra>"
    ),
    row=1, col=1
)

min_e = elec_df['Usage Rate ($/unit)'].min()
max_e = elec_df['Usage Rate ($/unit)'].max()
fig.add_hline(y=min_e, line_dash='dot', line_color='green', annotation_text=f"Min: ${min_e:.3f}", row=1, col=1)
fig.add_hline(y=max_e, line_dash='dot', line_color='purple', annotation_text=f"Max: ${max_e:.3f}", row=1, col=1)

# --- Top Right: Gas Usage Rate ---
gas_tl = build_timeline(gas_df, 'Usage Rate ($/unit)')
fig.add_trace(
    go.Scatter(x=gas_tl['Date'], y=gas_tl['Rate'], mode='lines', line=dict(color=gas_color, width=2.5), name='Gas Usage Rate', hoverinfo='skip'),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=gas_df['Period Start'], y=gas_df['Usage Rate ($/unit)'],
        mode='markers', marker=dict(color=gas_color, size=8),
        name='Gas Bill Data Point',
        customdata=np.stack((gas_df['Provider'], gas_df['Period Start'].dt.strftime('%b %d, %Y'), gas_df['Period End'].dt.strftime('%b %d, %Y'), gas_df['Total Usage'], gas_df['Total ($)']), axis=-1),
        hovertemplate="<b>Gas Bill</b><br><b>Provider:</b> %{customdata[0]}<br><b>Period:</b> %{customdata[1]} to %{customdata[2]}<br><b>Usage Rate:</b> $%{y:.5f}/MJ<br><b>Total Usage:</b> %{customdata[3]} MJ<br><b>Total Bill:</b> $%{customdata[4]:.2f}<extra></extra>"
    ),
    row=1, col=2
)

min_g = gas_df['Usage Rate ($/unit)'].min()
max_g = gas_df['Usage Rate ($/unit)'].max()
fig.add_hline(y=min_g, line_dash='dot', line_color='green', annotation_text=f"Min: ${min_g:.4f}", row=1, col=2)
fig.add_hline(y=max_g, line_dash='dot', line_color='purple', annotation_text=f"Max: ${max_g:.4f}", row=1, col=2)

# --- Bottom Left: Electricity Daily Supply Rate ---
elec_ds_tl = build_timeline(elec_df, 'Daily Supply Rate ($/day)')
fig.add_trace(
    go.Scatter(x=elec_ds_tl['Date'], y=elec_ds_tl['Rate'], mode='lines', line=dict(color=elec_color, width=2.5), name='Elec Supply Rate', hoverinfo='skip'),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=elec_df['Period Start'], y=elec_df['Daily Supply Rate ($/day)'],
        mode='markers', marker=dict(color=elec_color, size=8),
        name='Elec Supply Point',
        customdata=np.stack((elec_df['Provider'], elec_df['Period Start'].dt.strftime('%b %d, %Y'), elec_df['Period End'].dt.strftime('%b %d, %Y')), axis=-1),
        hovertemplate="<b>Electricity Supply Rate</b><br><b>Provider:</b> %{customdata[0]}<br><b>Period:</b> %{customdata[1]} to %{customdata[2]}<br><b>Daily Supply Rate:</b> $%{y:.2f}/day<extra></extra>"
    ),
    row=2, col=1
)

min_e_ds = elec_df['Daily Supply Rate ($/day)'].min()
max_e_ds = elec_df['Daily Supply Rate ($/day)'].max()
fig.add_hline(y=min_e_ds, line_dash='dot', line_color='green', annotation_text=f"Min: ${min_e_ds:.2f}", row=2, col=1)
fig.add_hline(y=max_e_ds, line_dash='dot', line_color='purple', annotation_text=f"Max: ${max_e_ds:.2f}", row=2, col=1)

# --- Bottom Right: Gas Daily Supply Rate ---
gas_ds_tl = build_timeline(gas_df, 'Daily Supply Rate ($/day)')
fig.add_trace(
    go.Scatter(x=gas_ds_tl['Date'], y=gas_ds_tl['Rate'], mode='lines', line=dict(color=gas_color, width=2.5), name='Gas Supply Rate', hoverinfo='skip'),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(
        x=gas_df['Period Start'], y=gas_df['Daily Supply Rate ($/day)'],
        mode='markers', marker=dict(color=gas_color, size=8),
        name='Gas Supply Point',
        customdata=np.stack((gas_df['Provider'], gas_df['Period Start'].dt.strftime('%b %d, %Y'), gas_df['Period End'].dt.strftime('%b %d, %Y')), axis=-1),
        hovertemplate="<b>Gas Supply Rate</b><br><b>Provider:</b> %{customdata[0]}<br><b>Period:</b> %{customdata[1]} to %{customdata[2]}<br><b>Daily Supply Rate:</b> $%{y:.2f}/day<extra></extra>"
    ),
    row=2, col=2
)

min_g_ds = gas_df['Daily Supply Rate ($/day)'].min()
max_g_ds = gas_df['Daily Supply Rate ($/day)'].max()
fig.add_hline(y=min_g_ds, line_dash='dot', line_color='green', annotation_text=f"Min: ${min_g_ds:.2f}", row=2, col=2)
fig.add_hline(y=max_g_ds, line_dash='dot', line_color='purple', annotation_text=f"Max: ${max_g_ds:.2f}", row=2, col=2)

fig.update_layout(
    title_text="Electricity & Gas Pricing Rates Overview (2024 - 2026)",
    title_x=0.5,
    title_font=dict(size=20),
    height=800,
    showlegend=False,
    template="plotly_white"
)
fig.update_yaxes(title_text="$/kWh", tickformat="$%.3f", row=1, col=1)
fig.update_yaxes(title_text="$/MJ", tickformat="$%.4f", row=1, col=2)
fig.update_yaxes(title_text="$/day", tickformat="$%.2f", row=2, col=1)
fig.update_yaxes(title_text="$/day", tickformat="$%.2f", row=2, col=2)

fig.write_html(output_dir / 'chart_rates_overview_interactive.html')
fig.show()

## 📈 3. Usage Rates Comparison (Electricity & Gas)

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Electricity Usage Rate ($/kWh)",
        "Gas Usage Rate ($/MJ)"
    ),
    vertical_spacing=0.1
)

# Top Panel: Electricity Usage Rate
fig.add_trace(
    go.Scatter(x=elec_tl['Date'], y=elec_tl['Rate'], mode='lines', line=dict(color='#1f77b4', width=2.5), name='Electricity Usage Rate ($/kWh)', hoverinfo='skip'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=elec_df['Period Start'], y=elec_df['Usage Rate ($/unit)'],
        mode='markers', marker=dict(color='#1f77b4', size=9),
        name='Electricity Bill Points',
        customdata=np.stack((elec_df['Provider'], elec_df['Period Start'].dt.strftime('%b %d, %Y'), elec_df['Period End'].dt.strftime('%b %d, %Y'), elec_df['Total Usage'], elec_df['Total ($)']), axis=-1),
        hovertemplate="<b>Electricity Bill Data Point</b><br><b>Provider:</b> %{customdata[0]}<br><b>Period:</b> %{customdata[1]} to %{customdata[2]}<br><b>Usage Rate:</b> $%{y:.4f}/kWh<br><b>Total Usage:</b> %{customdata[3]} kWh<br><b>Total Bill:</b> $%{customdata[4]:.2f}<extra></extra>"
    ),
    row=1, col=1
)

transition_date = '2025-05-26'
fig.add_vline(x=transition_date, line_dash='dash', line_color='darkorange', annotation_text="Provider Switch (EnergyAustralia → GloBird)", row=1, col=1)

# Bottom Panel: Gas Usage Rate
fig.add_trace(
    go.Scatter(x=gas_tl['Date'], y=gas_tl['Rate'], mode='lines', line=dict(color='#d62728', width=2.5), name='Gas Usage Rate ($/MJ)', hoverinfo='skip'),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=gas_df['Period Start'], y=gas_df['Usage Rate ($/unit)'],
        mode='markers', marker=dict(color='#d62728', size=9),
        name='Gas Bill Points',
        customdata=np.stack((gas_df['Provider'], gas_df['Period Start'].dt.strftime('%b %d, %Y'), gas_df['Period End'].dt.strftime('%b %d, %Y'), gas_df['Total Usage'], gas_df['Total ($)']), axis=-1),
        hovertemplate="<b>Gas Bill Data Point</b><br><b>Provider:</b> %{customdata[0]}<br><b>Period:</b> %{customdata[1]} to %{customdata[2]}<br><b>Usage Rate:</b> $%{y:.5f}/MJ<br><b>Total Usage:</b> %{customdata[3]} MJ<br><b>Total Bill:</b> $%{customdata[4]:.2f}<extra></extra>"
    ),
    row=2, col=1
)
fig.add_vline(x=transition_date, line_dash='dash', line_color='darkorange', annotation_text="Provider Switch (EnergyAustralia → GloBird)", row=2, col=1)

fig.update_layout(
    title_text="Electricity & Gas Usage Rates Over Time",
    title_x=0.5,
    title_font=dict(size=18),
    height=650,
    template="plotly_white",
    hovermode="closest"
)
fig.update_yaxes(title_text="Usage Rate ($/kWh)", tickformat="$%.4f", row=1, col=1)
fig.update_yaxes(title_text="Usage Rate ($/MJ)", tickformat="$%.5f", row=2, col=1)
fig.update_xaxes(title_text="Billing Period Date", row=2, col=1)

fig.write_html(output_dir / 'chart_usage_rates_interactive.html')
fig.show()

## 🔌 4. Daily Supply Rates Comparison

In [ ]:
fig = go.Figure()

# Electricity Daily Supply
fig.add_trace(
    go.Scatter(
        x=elec_ds_tl['Date'], y=elec_ds_tl['Rate'],
        mode='lines', line=dict(color='#1f77b4', width=2.5),
        name='Electricity Daily Supply Rate ($/day)',
        hoverinfo='skip'
    )
)
fig.add_trace(
    go.Scatter(
        x=elec_df['Period Start'], y=elec_df['Daily Supply Rate ($/day)'],
        mode='markers', marker=dict(color='#1f77b4', size=10, symbol='circle'),
        name='Electricity Points',
        customdata=np.stack((elec_df['Provider'], elec_df['Period Start'].dt.strftime('%b %d, %Y'), elec_df['Period End'].dt.strftime('%b %d, %Y'), elec_df['Daily Supply Charge ($)']), axis=-1),
        hovertemplate="<b>Electricity Daily Supply</b><br><b>Provider:</b> %{customdata[0]}<br><b>Period:</b> %{customdata[1]} to %{customdata[2]}<br><b>Daily Supply Rate:</b> $%{y:.2f}/day<br><b>Supply Charge Total:</b> $%{customdata[3]:.2f}<extra></extra>"
    )
)

# Gas Daily Supply
fig.add_trace(
    go.Scatter(
        x=gas_ds_tl['Date'], y=gas_ds_tl['Rate'],
        mode='lines', line=dict(color='#d62728', width=2.5),
        name='Gas Daily Supply Rate ($/day)',
        hoverinfo='skip'
    )
)
fig.add_trace(
    go.Scatter(
        x=gas_df['Period Start'], y=gas_df['Daily Supply Rate ($/day)'],
        mode='markers', marker=dict(color='#d62728', size=10, symbol='square'),
        name='Gas Points',
        customdata=np.stack((gas_df['Provider'], gas_df['Period Start'].dt.strftime('%b %d, %Y'), gas_df['Period End'].dt.strftime('%b %d, %Y'), gas_df['Daily Supply Charge ($)']), axis=-1),
        hovertemplate="<b>Gas Daily Supply</b><br><b>Provider:</b> %{customdata[0]}<br><b>Period:</b> %{customdata[1]} to %{customdata[2]}<br><b>Daily Supply Rate:</b> $%{y:.2f}/day<br><b>Supply Charge Total:</b> $%{customdata[3]:.2f}<extra></extra>"
    )
)

# Vertical switch line
fig.add_vline(x=transition_date, line_dash='dash', line_color='darkorange', annotation_text="Provider Switch (EnergyAustralia → GloBird)", annotation_position="top left")

fig.update_layout(
    title='Daily Supply Rate Comparison: Electricity vs. Gas ($/day)',
    title_x=0.5,
    title_font=dict(size=18),
    xaxis_title='Billing Period Date',
    yaxis_title='Daily Supply Rate ($/day)',
    height=550,
    template='plotly_white',
    hovermode='closest',
    legend=dict(yanchor="top", y=0.98, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)")
)
fig.update_yaxes(tickformat="$%.2f")

fig.write_html(output_dir / 'chart_daily_supply_rates_interactive.html')
fig.show()